# 01 · Define & Explore — cytokine signaling, target subunits, selectivity

**Standard slot:** *define & explore.* **For Project 13 this means:** understand IL-2 receptor
signaling, **choose which receptor subunits to engage** (e.g., IL-2Rβ + γc) and which to **spare**
(IL-2Rα/CD25), separate the subunits, write down the binder/selectivity metrics + cutoffs, and run a
deterministic **mock** mini-run as your "hello-world" (D0) — including a per-subunit **selectivity
profile**.

Run `00_setup.ipynb` first in this session. A real agonist campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## Why a de novo cytokine mimetic (and why selectivity)

Native **IL-2** is a powerful but toxic, unstable drug. Its receptor has three chains:
- **IL-2Rα (CD25)** — high-affinity **capture** chain; **does not signal**. Engaging it is what makes
  native IL-2 hit CD25-high regulatory T cells (Tregs) and drives vascular-leak toxicity.
- **IL-2Rβ (CD122)** + **γc (CD132)** — the **signaling pair**: dimerizing them juxtaposes JAK1/JAK3
  → **STAT5** phosphorylation → effector T/NK activation.

The **Neo-2/15** paradigm (Silva et al. 2019): a hyperstable de novo mini-protein that engages **β + γc
but has no α site** → a **βγ-biased agonist** that keeps the useful signaling, spares Tregs, and is far
more stable than IL-2. This notebook sets up that design decision and the way we will *prove* it in
silico: model each design against **each subunit separately** and report the selectivity profile.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the mimetic | thermostability / ΔG |
| **pae_interaction (per subunit)** | Å | AF2-Multimer error across the mimetic–**subunit** interface (the key metric, computed for **each** of α/β/γc) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **selectivity margin** | Å | `min(pae over spared) − max(pae over engaged)`; larger = cleaner | agonism (binding ≠ signaling) |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs (applied to the **engaged** subunits): **scRMSD ≤ 2.5, pLDDT ≥ 80,
pae_interaction ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.** `pae_interaction` is the single most important
metric — but a low value is *confidence*, **not** affinity, and a **selective binder is not an
agonist**: only a cell pSTAT5 assay decides signaling. A passing, selective design is a **hypothesis**
until per-subunit SPR **and** a STAT-phosphorylation assay.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target prep + per-subunit hotspots

The design target is the **IL-2Rβ + γc signaling surface** (you must bridge **both** chains to dimerize
the receptor → signal), and you also keep the **IL-2Rα** subunit aside to model selectivity *against*.
Fetch the candidate complex with `data/download_data.py` (**2B5I — verify on RCSB**), identify the
chains, **split** IL-2Rα / IL-2Rβ / γc into separate targets, and read the IL-2 contact residues off
each as your per-subunit hotspots.

Below we just *declare* EXAMPLE hotspots so the notebook runs end-to-end; **replace them with the
residues you derive from the actual IL-2/IL-2R interface** (numbering depends on the PDB you verify).

In [ ]:
import cytokine_tools as ct

TARGET = "IL2R_beta_gamma"           # the β+γc signaling surface (you produce this from 2B5I)
# EXAMPLE signaling-face hotspots on IL-2Rβ (chain B) and γc (chain C) — VERIFY/REPLACE from the
# IL-2/IL-2R interface (data/README.md). These are placeholders so the plumbing runs.
HOTSPOTS = ct.parse_hotspots("B41,B42,C100,C102")   # EXAMPLE_DATA placeholder residues

# The selectivity goal (Neo-2/15-style): ENGAGE the signaling pair, SPARE the capture chain.
ENGAGE = ct.ENGAGE_DEFAULT           # ("IL2Rb", "gammaC")  -> dimerize these to signal
SPARE  = ct.SPARE_DEFAULT            # ("IL2Ra",)           -> CD25, the chain we want to AVOID
print("subunits     :", ct.SUBUNITS)
print("target       :", TARGET)
print("hotspots     :", HOTSPOTS, " (EXAMPLE — replace with your verified β/γc residues)")
print("ENGAGE (signal):", ENGAGE, "  SPARE (capture/CD25):", SPARE)

## 2 · Mock hello-world: a tiny agonist mini-run + per-subunit selectivity

`scripts/cytokine_tools.py` exposes the design paradigms behind one API
(`generate_agonists_rfdiffusion(...)`, `generate_agonists_bindcraft(...)`), the per-subunit AF2-Multimer
scorer (`af2_multimer(seq, subunit=...)`), and the project's twist —
`selectivity_profile(...)` (model vs **each** subunit → engages βγ but not α?). The **mock** backend is
deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as real** — they
are `SYNTHETIC` by construction (and there is no fabricated EC50/K_D anywhere).

In [ ]:
# A few designs, scored against EACH subunit, with a selectivity call. All numbers are SYNTHETIC.
rf = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
ct.score_designs(rf, tool="mock", engage=ENGAGE, spare=SPARE)

d = rf[0]
prof = ct.selectivity_profile(d, tool="mock", engage=ENGAGE, spare=SPARE)
print("example RFdiffusion agonist design:")
print("  id    :", d.design_id)
print("  len   :", d.length, "aa")
print("  seq   :", d.sequence)
print("  plddt :", d.plddt, " scrmsd:", d.scrmsd, " sc:", d.shape_complementarity, " (SYNTHETIC)")
print("  per-subunit pae (SELECTIVITY PROFILE):", prof["pae_by_subunit"], "(SYNTHETIC)")
print("  engaged_ok:", prof["engaged_ok"], " spared_ok:", prof["spared_ok"],
      " margin:", prof["selectivity_margin"], " -> selective:", prof["selective"])
print("  synthetic flag:", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'rfdiffusion'/'bindcraft'/'af2' on Colab (A100). See MANUAL.md §2.")

## 3 · Read the selectivity profile

For a **βγ-biased** agonist you want **low** `pae` to IL-2Rβ and γc (confidently engaged → can
dimerize → signal) and **high** `pae` to IL-2Rα (the capture chain is spared). The `selective` flag
combines: engaged-OK **and** spared-OK **and** a selectivity margin above threshold. A design with low
`pae` to **all three** subunits is *not* selective — it behaves like toxic native IL-2.

In [ ]:
for b in rf[:3]:
    p = ct.selectivity_profile(b, tool="mock", engage=ENGAGE, spare=SPARE)
    eng = {s: p["pae_by_subunit"][s] for s in ENGAGE}
    spr = {s: p["pae_by_subunit"][s] for s in SPARE}
    print(f"{b.design_id}: engage(low?) {eng}  spare(high?) {spr}  "
          f"margin={p['selectivity_margin']}  selective={p['selective']}  (SYNTHETIC)")
print("\nNOTE: SELECTIVE in silico is a hypothesis about which subunits are ENGAGED — not proof of")
print("agonism. Binding != signaling; the cell pSTAT5 assay (notebook 05) decides it.")

## Visualize a mimetic–receptor complex (py3Dmol)

Use this to eyeball a predicted mimetic–receptor complex once you have a real PDB (from AF2-Multimer).
For agonism, check that the mini-protein bridges **both** β and γc.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex_beta_gamma.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] IL-2/IL-2R accession verified on RCSB (2B5I is a candidate); chains identified (IL-2 vs α/β/γc).
- [ ] Receptor subunits **separated** (α, β, γc) + per-subunit **hotspot lists** (derived from the interface, not invented).
- [ ] **Target-subunit/selectivity choice written down** (engage β+γc, spare α — and *why*: reduce Treg/vascular-leak toxicity).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (incl. binding ≠ signaling).
- [ ] Reproduced mock mini-run with the **per-subunit selectivity profile** printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the agonist campaign engaging the chosen receptor surfaces.